# Kankor Retrieval Benchmark: `BAAI/bge-m3`

This notebook benchmarks the canonical page corpus with the new `bge_m3` dense backend.

It supports:
- BGE-M3 dense retrieval
- BGE-M3 dense + lexical + RRF
- optional reranker evaluation
- reranker candidate-pool sweeps for overnight runs
- promotion analysis for whether a relevant answer found in the candidate pool gets moved into top 3
- optional direct comparison against the `text-embedding-3-large` baseline


In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'packages' / 'rag_core').exists() and (candidate / 'apps' / 'api').exists():
            return candidate
    raise RuntimeError('Could not resolve repo root from notebook location.')

REPO_ROOT = resolve_repo_root(Path.cwd())
DOTENV_PATH = REPO_ROOT / 'docker' / '.env'
if DOTENV_PATH.exists():
    for raw_line in DOTENV_PATH.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

sys.path.insert(0, str(REPO_ROOT / 'packages' / 'rag_core' / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'apps' / 'api' / 'src'))

from rag_core.eval.retrieval_benchmark import (
    benchmark_dense_retrieval,
    benchmark_hybrid_retrieval,
    benchmark_hybrid_reranked_retrieval,
    load_qrels,
    load_query_suite,
)
from rag_core.impl.embeddings_bge_m3 import BGEM3Embedder
from rag_core.impl.embeddings_openai import OpenAIEmbedder
from rag_core.impl.reranker_onnx import ONNXSequenceClassificationReranker
from rag_core.impl.vector_faiss import FaissVectorStore
from rag_core.rag.retrieval import DenseRetriever, LexicalRetriever, PageRetrievalEngine, RRFFusionPolicy


In [ ]:
QUERY_SUITE_NAME = 'chunking_pages_suite_v2'
QRELS_NAME = 'chunking_pages_qrels_v2'
RUN_OPENAI_BASELINE = False
RUN_RERANKER = True
QUERY_LIMIT = None

K_VALUES = (1, 3, 5, 10)
QUERY_BATCH_SIZE = 64
RRF_K = 20
DENSE_MIN_SCORE = 0.15
LEXICAL_MIN_SCORE = 0.01
RERANKER_TARGET_TOP_K = 3
RERANKER_CANDIDATE_POOL_SIZES = (3, 5, 8, 10, 15, 20)
RERANKER_CANDIDATE_POOL_SIZE = 8
SAVE_RESULTS = True
RESULTS_ROOT = REPO_ROOT / 'data' / 'experiments' / 'benchmark_kankor_bge_m3'
RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

BGE_INDEX_DIR = REPO_ROOT / 'data' / 'index' / 'kankor_bge_m3_full'
OPENAI_INDEX_DIR = REPO_ROOT / 'data' / 'index' / 'kankor_openai_full'
QUERY_SUITE_PATH = REPO_ROOT / 'data' / 'query_suites' / f'{QUERY_SUITE_NAME}.jsonl'
QRELS_PATH = REPO_ROOT / 'data' / 'query_suites' / f'{QRELS_NAME}.jsonl'

RERANKER_MODEL_ID = os.getenv('RAG_RERANKER_MODEL_ID', 'onnx-community/gte-multilingual-reranker-base')
RERANKER_MODEL_REVISION = os.getenv('RAG_RERANKER_MODEL_REVISION') or None
RERANKER_MAX_LENGTH = int(os.getenv('RAG_RERANKER_MAX_LENGTH', '256'))
RERANKER_BATCH_SIZE = int(os.getenv('RAG_RERANKER_BATCH_SIZE', '8'))

assert (BGE_INDEX_DIR / 'index.faiss').exists(), f'Missing BGE index: {BGE_INDEX_DIR / "index.faiss"}'
assert (BGE_INDEX_DIR / 'metadata.jsonl').exists(), f'Missing BGE metadata: {BGE_INDEX_DIR / "metadata.jsonl"}'
assert QUERY_SUITE_PATH.exists(), f'Missing query suite: {QUERY_SUITE_PATH}'
assert QRELS_PATH.exists(), f'Missing qrels: {QRELS_PATH}'
if RUN_OPENAI_BASELINE:
    assert (OPENAI_INDEX_DIR / 'index.faiss').exists(), f'Missing OpenAI index: {OPENAI_INDEX_DIR / "index.faiss"}'
    assert (OPENAI_INDEX_DIR / 'metadata.jsonl').exists(), f'Missing OpenAI metadata: {OPENAI_INDEX_DIR / "metadata.jsonl"}'

display({
    'repo_root': str(REPO_ROOT),
    'bge_index_dir': str(BGE_INDEX_DIR),
    'openai_index_dir': str(OPENAI_INDEX_DIR),
    'query_suite': str(QUERY_SUITE_PATH),
    'qrels': str(QRELS_PATH),
    'run_openai_baseline': RUN_OPENAI_BASELINE,
    'run_reranker': RUN_RERANKER,
    'reranker_target_top_k': RERANKER_TARGET_TOP_K,
    'reranker_candidate_pool_sizes': RERANKER_CANDIDATE_POOL_SIZES,
    'default_reranker_candidate_pool_size': RERANKER_CANDIDATE_POOL_SIZE,
    'save_results': SAVE_RESULTS,
    'results_root': str(RESULTS_ROOT),
    'run_label': RUN_LABEL,
})


In [ ]:
queries = load_query_suite(QUERY_SUITE_PATH, allowed_intents={'grounded_textbook'})
qrels_by_query = load_qrels(QRELS_PATH)
if QUERY_LIMIT is not None:
    queries = queries[: int(QUERY_LIMIT)]

def build_stack(index_dir: Path, *, embedder):
    vector_store = FaissVectorStore.load(
        index_path=index_dir / 'index.faiss',
        metadata_path=index_dir / 'metadata.jsonl',
    )
    dense = DenseRetriever(embedder=embedder, vector_store=vector_store, min_score=DENSE_MIN_SCORE)
    lexical = LexicalRetriever(vector_store=vector_store, min_score=LEXICAL_MIN_SCORE)
    engine = PageRetrievalEngine(
        dense_retriever=dense,
        lexical_retriever=lexical,
        fusion_policy=RRFFusionPolicy(rrf_k=RRF_K),
    )
    return vector_store, dense, engine

bge_embedder = BGEM3Embedder(
    model_name=os.getenv('RAG_BGE_M3_MODEL_ID', 'BAAI/bge-m3'),
    batch_size=int(os.getenv('RAG_BGE_M3_BATCH_SIZE', '32')),
    use_fp16=(os.getenv('RAG_BGE_M3_USE_FP16') or '').strip().lower() == 'true' if os.getenv('RAG_BGE_M3_USE_FP16') is not None else None,
    device=os.getenv('RAG_BGE_M3_DEVICE') or None,
    max_length=int(os.getenv('RAG_BGE_M3_MAX_LENGTH', '8192')),
)
_, bge_dense_retriever, bge_hybrid_engine = build_stack(BGE_INDEX_DIR, embedder=bge_embedder)

openai_dense_retriever = None
openai_hybrid_engine = None
if RUN_OPENAI_BASELINE:
    openai_embedder = OpenAIEmbedder(
        model_name=os.getenv('RAG_OPENAI_EMBEDDING_MODEL_ID', 'text-embedding-3-large'),
        api_key=os.getenv('RAG_OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY'),
        base_url=os.getenv('RAG_OPENAI_BASE_URL') or None,
        dimensions=int(os.getenv('RAG_OPENAI_EMBEDDING_DIMENSIONS')) if os.getenv('RAG_OPENAI_EMBEDDING_DIMENSIONS') else None,
        timeout_seconds=float(os.getenv('RAG_OPENAI_TIMEOUT_SECONDS', '120.0')),
    )
    _, openai_dense_retriever, openai_hybrid_engine = build_stack(OPENAI_INDEX_DIR, embedder=openai_embedder)

reranker = None
if RUN_RERANKER:
    reranker = ONNXSequenceClassificationReranker(
        model_name=RERANKER_MODEL_ID,
        model_revision=RERANKER_MODEL_REVISION,
        max_length=RERANKER_MAX_LENGTH,
        batch_size=RERANKER_BATCH_SIZE,
    )
    reranker.warmup()

display({'queries': len(queries), 'qrels': len(qrels_by_query)})


In [ ]:
RERANKER_CANDIDATE_POOL_SIZES = tuple(
    sorted(
        {
            max(RERANKER_TARGET_TOP_K, int(value))
            for value in RERANKER_CANDIDATE_POOL_SIZES
        }
    )
)
DEFAULT_RERANKER_CANDIDATE_POOL_SIZE = max(RERANKER_TARGET_TOP_K, int(RERANKER_CANDIDATE_POOL_SIZE))
BASE_K_VALUES = tuple(sorted({*K_VALUES, RERANKER_TARGET_TOP_K}))
DEFAULT_RERANK_K_VALUES = tuple(sorted({*BASE_K_VALUES, DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}))

bge_dense_results = benchmark_dense_retrieval(
    queries=queries,
    qrels_by_query=qrels_by_query,
    dense_retriever=bge_dense_retriever,
    k_values=BASE_K_VALUES,
    query_batch_size=QUERY_BATCH_SIZE,
)
bge_hybrid_results = benchmark_hybrid_retrieval(
    queries=queries,
    qrels_by_query=qrels_by_query,
    retrieval_engine=bge_hybrid_engine,
    k_values=BASE_K_VALUES,
    query_batch_size=QUERY_BATCH_SIZE,
)
bge_reranked_results = None
bge_reranked_results_by_pool = {}
if RUN_RERANKER:
    print(
        f'Running default reranked benchmark for candidate_pool_size={DEFAULT_RERANKER_CANDIDATE_POOL_SIZE} '
        f'with k_values={DEFAULT_RERANK_K_VALUES} ...'
    )
    bge_reranked_results = benchmark_hybrid_reranked_retrieval(
        queries=queries,
        qrels_by_query=qrels_by_query,
        retrieval_engine=bge_hybrid_engine,
        reranker=reranker,
        k_values=DEFAULT_RERANK_K_VALUES,
        reranker_candidate_pool_size=DEFAULT_RERANKER_CANDIDATE_POOL_SIZE,
        query_batch_size=QUERY_BATCH_SIZE,
    )
    bge_reranked_results_by_pool[DEFAULT_RERANKER_CANDIDATE_POOL_SIZE] = bge_reranked_results
    for candidate_pool_size in RERANKER_CANDIDATE_POOL_SIZES:
        if candidate_pool_size == DEFAULT_RERANKER_CANDIDATE_POOL_SIZE:
            continue
        sweep_k_values = tuple(sorted({RERANKER_TARGET_TOP_K, 5, candidate_pool_size}))
        print(
            f'Running reranker sweep for candidate_pool_size={candidate_pool_size} '
            f'with k_values={sweep_k_values} ...'
        )
        bge_reranked_results_by_pool[candidate_pool_size] = benchmark_hybrid_reranked_retrieval(
            queries=queries,
            qrels_by_query=qrels_by_query,
            retrieval_engine=bge_hybrid_engine,
            reranker=reranker,
            k_values=sweep_k_values,
            reranker_candidate_pool_size=candidate_pool_size,
            query_batch_size=QUERY_BATCH_SIZE,
        )

openai_dense_results = None
openai_hybrid_results = None
if RUN_OPENAI_BASELINE:
    openai_dense_results = benchmark_dense_retrieval(
        queries=queries,
        qrels_by_query=qrels_by_query,
        dense_retriever=openai_dense_retriever,
        k_values=BASE_K_VALUES,
        query_batch_size=QUERY_BATCH_SIZE,
    )
    openai_hybrid_results = benchmark_hybrid_retrieval(
        queries=queries,
        qrels_by_query=qrels_by_query,
        retrieval_engine=openai_hybrid_engine,
        k_values=BASE_K_VALUES,
        query_batch_size=QUERY_BATCH_SIZE,
    )

def aggregate_frame(result, prefix: str):
    frame = pd.DataFrame.from_dict(result['aggregate_by_k'], orient='index').set_index('k')
    return frame.add_prefix(prefix)

comparison = aggregate_frame(bge_dense_results, 'bge_dense_').join(aggregate_frame(bge_hybrid_results, 'bge_hybrid_'))
if bge_reranked_results is not None:
    comparison = comparison.join(aggregate_frame(bge_reranked_results, 'bge_reranked_'))
if openai_dense_results is not None:
    comparison = comparison.join(aggregate_frame(openai_dense_results, 'openai_dense_'))
if openai_hybrid_results is not None:
    comparison = comparison.join(aggregate_frame(openai_hybrid_results, 'openai_hybrid_'))
comparison.loc[list(K_VALUES)]


In [ ]:
def per_query_frame(result, system_name: str, *, candidate_pool_size: int | None = None):
    frame = pd.DataFrame(result['per_query']).copy()
    frame['system'] = system_name
    frame['expected_subject'] = frame['expected_subjects'].apply(lambda values: values[0] if values else '')
    frame['candidate_pool_size'] = candidate_pool_size
    return frame

frames = [
    per_query_frame(bge_dense_results, 'bge_dense'),
    per_query_frame(bge_hybrid_results, 'bge_hybrid'),
]
if bge_reranked_results is not None:
    frames.append(per_query_frame(bge_reranked_results, f'bge_reranked_pool_{DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}', candidate_pool_size=DEFAULT_RERANKER_CANDIDATE_POOL_SIZE))
for candidate_pool_size, result in bge_reranked_results_by_pool.items():
    system_name = f'bge_reranked_pool_{candidate_pool_size}'
    if system_name == f'bge_reranked_pool_{DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}':
        continue
    frames.append(per_query_frame(result, system_name, candidate_pool_size=candidate_pool_size))
if openai_dense_results is not None:
    frames.append(per_query_frame(openai_dense_results, 'openai_dense'))
if openai_hybrid_results is not None:
    frames.append(per_query_frame(openai_hybrid_results, 'openai_hybrid'))
all_per_query = pd.concat(frames, ignore_index=True)

def aggregate_for_display(result, *, label: str, k: int) -> dict:
    aggregate = result['aggregate_by_k'][str(k)]
    return {
        'system': label,
        'k': k,
        'mean_hit_rate_at_k': aggregate['mean_hit_rate_at_k'],
        'mean_mrr_at_k': aggregate['mean_mrr_at_k'],
        'mean_recall_at_k': aggregate['mean_recall_at_k'],
        'mean_ndcg_at_k': aggregate['mean_ndcg_at_k'],
        'query_count_evaluated': aggregate['query_count_evaluated'],
    }

reranker_sweep_rows = []
promotion_detail_frames = []
hybrid_per_query = per_query_frame(bge_hybrid_results, 'bge_hybrid')
hybrid_pool_rank_columns = [
    f'first_relevant_rank_at_{pool_size}'
    for pool_size in RERANKER_CANDIDATE_POOL_SIZES
    if pool_size != RERANKER_TARGET_TOP_K
]
hybrid_rank_columns = ['query_id', 'query', 'language', 'expected_subject', f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}', *hybrid_pool_rank_columns]
hybrid_rank_view = hybrid_per_query[hybrid_rank_columns].copy()

for candidate_pool_size, result in bge_reranked_results_by_pool.items():
    label = f'bge_reranked_pool_{candidate_pool_size}'
    reranked_per_query = per_query_frame(result, label, candidate_pool_size=candidate_pool_size)
    top3_rank_column = f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}'
    pool_rank_column = f'first_relevant_rank_at_{candidate_pool_size}'
    reranked_rank_view = reranked_per_query[['query_id', top3_rank_column, pool_rank_column]].rename(
        columns={
            top3_rank_column: 'reranked_first_relevant_rank_at_top3',
            pool_rank_column: 'reranked_first_relevant_rank_at_pool',
        }
    )
    promotion_frame = hybrid_rank_view.merge(reranked_rank_view, on='query_id', how='left')
    promotion_frame['candidate_pool_size'] = candidate_pool_size
    promotion_frame['relevant_in_hybrid_pool'] = promotion_frame[f'first_relevant_rank_at_{candidate_pool_size}'].notna()
    promotion_frame['relevant_in_hybrid_top3'] = promotion_frame[f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}'].notna()
    promotion_frame['relevant_in_reranked_top3'] = promotion_frame['reranked_first_relevant_rank_at_top3'].notna()
    promotion_frame['promoted_into_top3'] = (
        promotion_frame['relevant_in_hybrid_pool']
        & ~promotion_frame['relevant_in_hybrid_top3']
        & promotion_frame['relevant_in_reranked_top3']
    )
    promotion_frame['lost_from_top3'] = (
        promotion_frame['relevant_in_hybrid_top3']
        & ~promotion_frame['relevant_in_reranked_top3']
    )
    promotion_frame['net_top3_gain'] = promotion_frame['relevant_in_reranked_top3'].astype(int) - promotion_frame['relevant_in_hybrid_top3'].astype(int)
    promotion_detail_frames.append(promotion_frame)

    candidate_mask = promotion_frame['relevant_in_hybrid_pool']
    promotion_candidate_mask = promotion_frame['relevant_in_hybrid_pool'] & ~promotion_frame['relevant_in_hybrid_top3']
    rerank_ms = pd.DataFrame(result['retrieval_rows'])['rerank_ms'] if result['retrieval_rows'] else pd.Series(dtype='float64')
    aggregate_top3 = result['aggregate_by_k'][str(RERANKER_TARGET_TOP_K)]
    aggregate_top5 = result['aggregate_by_k'][str(5)]
    reranker_sweep_rows.append({
        'candidate_pool_size': candidate_pool_size,
        'mean_hit_rate_at_3': aggregate_top3['mean_hit_rate_at_k'],
        'mean_mrr_at_3': aggregate_top3['mean_mrr_at_k'],
        'mean_recall_at_3': aggregate_top3['mean_recall_at_k'],
        'mean_hit_rate_at_5': aggregate_top5['mean_hit_rate_at_k'],
        'mean_mrr_at_5': aggregate_top5['mean_mrr_at_k'],
        'mean_ndcg_at_5': aggregate_top5['mean_ndcg_at_k'],
        'mean_rerank_ms': float(rerank_ms.mean()) if not rerank_ms.empty else None,
        'median_rerank_ms': float(rerank_ms.median()) if not rerank_ms.empty else None,
        'queries_with_relevant_in_pool': int(candidate_mask.sum()),
        'queries_with_relevant_in_pool_and_not_top3': int(promotion_candidate_mask.sum()),
        'pool_to_top3_conversion_rate': float(promotion_frame.loc[candidate_mask, 'relevant_in_reranked_top3'].mean()) if candidate_mask.any() else None,
        'promotion_rate_given_not_hybrid_top3': float(promotion_frame.loc[promotion_candidate_mask, 'promoted_into_top3'].mean()) if promotion_candidate_mask.any() else None,
        'promoted_query_count': int(promotion_frame['promoted_into_top3'].sum()),
        'lost_top3_query_count': int(promotion_frame['lost_from_top3'].sum()),
        'net_top3_gain': int(promotion_frame['net_top3_gain'].sum()),
    })

reranker_sweep_summary = pd.DataFrame(reranker_sweep_rows).sort_values('candidate_pool_size') if reranker_sweep_rows else pd.DataFrame()
promotion_details = pd.concat(promotion_detail_frames, ignore_index=True) if promotion_detail_frames else pd.DataFrame()

language_slice = (
    all_per_query.groupby(['system', 'language'], dropna=False)
    .agg(
        query_count=('query_id', 'count'),
        mean_hit_rate_at_5=('hit_rate_at_5', 'mean'),
        mean_mrr_at_5=('mrr_at_5', 'mean'),
        mean_recall_at_5=('recall_at_5', 'mean'),
    )
    .reset_index()
    .sort_values(['system', 'language'])
)

subject_failure_slice = (
    all_per_query.groupby(['system', 'expected_subject'], dropna=False)
    .agg(
        query_count=('query_id', 'count'),
        fail_count_at_5=('hit_rate_at_5', lambda values: int((1.0 - values).sum())),
        fail_rate_at_5=('hit_rate_at_5', lambda values: float(1.0 - values.mean())),
        miss_top10_count=('hit_rate_at_10', lambda values: int((1.0 - values).sum())),
        miss_top10_rate=('hit_rate_at_10', lambda values: float(1.0 - values.mean())),
        mean_hit_rate_at_5=('hit_rate_at_5', 'mean'),
        mean_mrr_at_5=('mrr_at_5', 'mean'),
        mean_recall_at_5=('recall_at_5', 'mean'),
    )
    .reset_index()
    .sort_values(['system', 'fail_rate_at_5'], ascending=[True, False])
)

failure_rows = all_per_query[all_per_query['hit_rate_at_5'] < 1.0].sort_values(['system', 'language', 'expected_subject', 'query_id'])
aggregate_snapshot = pd.DataFrame([
    aggregate_for_display(bge_dense_results, label='bge_dense', k=5),
    aggregate_for_display(bge_hybrid_results, label='bge_hybrid', k=5),
    *[
        aggregate_for_display(result, label=f'bge_reranked_pool_{candidate_pool_size}', k=5)
        for candidate_pool_size, result in sorted(bge_reranked_results_by_pool.items())
    ],
])

display(aggregate_snapshot)
display(reranker_sweep_summary)
display(language_slice)
display(subject_failure_slice.head(30))
display(failure_rows[['system', 'query_id', 'language', 'expected_subject', 'query', 'first_relevant_rank_at_10']].head(100))
if not promotion_details.empty:
    promoted_examples = promotion_details[promotion_details['promoted_into_top3']].sort_values(['candidate_pool_size', 'language', 'expected_subject', 'query_id'])
    display(promoted_examples[['candidate_pool_size', 'query_id', 'language', 'expected_subject', 'query', f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}', 'reranked_first_relevant_rank_at_top3']].head(100))

summary_rows = []
if openai_dense_results is not None:
    bge_hit5 = comparison.loc[5, 'bge_dense_mean_hit_rate_at_k']
    openai_hit5 = comparison.loc[5, 'openai_dense_mean_hit_rate_at_k']
    bge_mrr5 = comparison.loc[5, 'bge_dense_mean_mrr_at_k']
    openai_mrr5 = comparison.loc[5, 'openai_dense_mean_mrr_at_k']
    bge_ps_hit5 = language_slice[(language_slice['system'] == 'bge_dense') & (language_slice['language'] == 'ps')]['mean_hit_rate_at_5'].iloc[0]
    openai_ps_hit5 = language_slice[(language_slice['system'] == 'openai_dense') & (language_slice['language'] == 'ps')]['mean_hit_rate_at_5'].iloc[0]
    summary_rows.append({
        'comparison': 'bge_dense_vs_openai_dense',
        'delta_hit_rate_at_5': bge_hit5 - openai_hit5,
        'delta_mrr_at_5': bge_mrr5 - openai_mrr5,
        'delta_pashto_hit_rate_at_5': bge_ps_hit5 - openai_ps_hit5,
        'promotion_candidate': bool((bge_hit5 > openai_hit5) and (bge_ps_hit5 > openai_ps_hit5)),
    })
baseline_summary = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame()
display(baseline_summary)

saved_paths = {}
if SAVE_RESULTS:
    run_output_dir = RESULTS_ROOT / RUN_LABEL
    run_output_dir.mkdir(parents=True, exist_ok=True)
    comparison.loc[list(K_VALUES)].to_csv(run_output_dir / 'aggregate_comparison.csv')
    aggregate_snapshot.to_csv(run_output_dir / 'aggregate_snapshot.csv', index=False)
    all_per_query.to_csv(run_output_dir / 'all_per_query.csv', index=False)
    language_slice.to_csv(run_output_dir / 'language_slice.csv', index=False)
    subject_failure_slice.to_csv(run_output_dir / 'subject_failure_slice.csv', index=False)
    failure_rows.to_csv(run_output_dir / 'failure_rows.csv', index=False)
    baseline_summary.to_csv(run_output_dir / 'baseline_summary.csv', index=False)
    if not reranker_sweep_summary.empty:
        reranker_sweep_summary.to_csv(run_output_dir / 'reranker_sweep_summary.csv', index=False)
    if not promotion_details.empty:
        promotion_details.to_csv(run_output_dir / 'reranker_promotion_details.csv', index=False)
    saved_paths = {
        'run_output_dir': str(run_output_dir),
        'aggregate_comparison': str(run_output_dir / 'aggregate_comparison.csv'),
        'reranker_sweep_summary': str(run_output_dir / 'reranker_sweep_summary.csv'),
        'reranker_promotion_details': str(run_output_dir / 'reranker_promotion_details.csv'),
    }
display(saved_paths)
